# Análise RD do sweep módulo-a-módulo de WSiLU

Este notebook analisa os resultados gerados por `DCVC/run_wsilu_module_sweep.py` em `coding_outputs/module_sweep`.

Objetivos:

1. Carregar o baseline `baseline_wsilu4.json` e todos os experimentos no padrão `<modulo>__<variante>.json`.
2. Calcular o aumento de BD-rate de cada variante de WSiLU em cada módulo, mantendo os demais módulos em `wsilu4`.
3. Montar uma tabela com módulos intra (`i.*`) primeiro e módulos inter (`p.*`) depois.
4. Plotar curvas RD por módulo, comparando baseline e variantes.

> Convenção: BD-rate positivo significa aumento de bitrate em relação ao baseline; BD-rate negativo indica melhoria.


In [ ]:
# Se o pacote não estiver instalado no ambiente do notebook, execute esta célula.
%pip install bjontegaard


In [ ]:
from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from bjontegaard import bd_rate

# -----------------------------------------------------------------------------
# Configuração geral
# -----------------------------------------------------------------------------
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

RESULTS_DIR = REPO_ROOT / "coding_outputs" / "module_sweep"
ANCHOR_NAME = "baseline_wsilu4"
FRAME_TYPE = "all"  # opções usuais: "all", "i", "p"
BD_RATE_METHOD = "akima"

VARIANT_ORDER = [
    "lut_asyn_4int_128entries",
    "lut_asyn_4int_256entries",
    "lut_asyn_4int_512entries",
]

VARIANT_LABELS = {
    "lut_asyn_4int_128entries": "LUT 128",
    "lut_asyn_4int_256entries": "LUT 256",
    "lut_asyn_4int_512entries": "LUT 512",
}

MODULE_ORDER = [
    # Intra / image model
    "i.enc",
    "i.dec",
    "i.hyper_enc",
    "i.hyper_dec",
    "i.y_prior_fusion",
    "i.y_spatial_prior_adaptor_1",
    "i.y_spatial_prior_adaptor_2",
    "i.y_spatial_prior_adaptor_3",
    "i.y_spatial_prior",
    # Inter / video model
    "p.feature_extractor",
    "p.encoder",
    "p.decoder",
    "p.recon_generation_net",
    "p.hyper_encoder",
    "p.hyper_decoder",
    "p.y_prior_fusion",
    "p.y_spatial_prior",
    "p.feature_adaptor_i",
    "p.temporal_prior_encoder",
]

PLOT_STYLE = {
    ANCHOR_NAME: {"color": "#000000", "marker": "o", "linestyle": "--", "linewidth": 2.4},
    "lut_asyn_4int_128entries": {"color": "#1f77b4", "marker": "s", "linestyle": "-", "linewidth": 2.0},
    "lut_asyn_4int_256entries": {"color": "#d62728", "marker": "^", "linestyle": "-", "linewidth": 2.0},
    "lut_asyn_4int_512entries": {"color": "#2ca02c", "marker": "D", "linestyle": "-", "linewidth": 2.0},
}

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

print(f"RESULTS_DIR = {RESULTS_DIR}")


In [ ]:
def metrics_json_to_df(src, experiment):
    """Converte o JSON de saída do test_video.py em um DataFrame tidy."""
    src = Path(src)
    with src.open("r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []
    for dataset_name, sequences in data.items():
        for seq_name, rates in sequences.items():
            for rate_key, metrics in rates.items():
                row = {
                    "experiment": experiment,
                    "dataset": dataset_name,
                    "sequence": seq_name,
                    "rate_key": rate_key,
                }
                row.update(metrics)
                rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    if "rate_idx" in df.columns:
        df["rate_idx"] = pd.to_numeric(df["rate_idx"], errors="coerce").astype("Int64")
    else:
        df["rate_idx"] = pd.to_numeric(df["rate_key"], errors="coerce").astype("Int64")
    return df


def parse_experiment_name(name):
    """Retorna (module, variant) a partir de '<module>__<variant>'."""
    if name == ANCHOR_NAME:
        return None, None
    if "__" not in name:
        return None, None
    module, variant = name.split("__", 1)
    return module, variant


def discover_result_files(results_dir=RESULTS_DIR):
    results_dir = Path(results_dir)
    if not results_dir.exists():
        raise FileNotFoundError(f"Diretório não encontrado: {results_dir}")

    files = sorted(results_dir.glob("*.json"))
    files = [p for p in files if p.name != "manifest.json"]
    anchor = results_dir / f"{ANCHOR_NAME}.json"
    if anchor not in files:
        raise FileNotFoundError(f"Baseline não encontrado: {anchor}")
    return files


def load_all_results(result_files):
    frames = []
    experiment_meta = []
    for path in result_files:
        experiment = path.stem
        module, variant = parse_experiment_name(experiment)
        df = metrics_json_to_df(path, experiment=experiment)
        if df.empty:
            print(f"[WARN] Resultado vazio ignorado: {path}")
            continue
        df["module"] = module
        df["variant"] = variant
        df["path"] = str(path)
        frames.append(df)
        experiment_meta.append({"experiment": experiment, "module": module, "variant": variant, "path": path})

    if not frames:
        raise ValueError("Nenhum resultado válido foi carregado.")
    return pd.concat(frames, ignore_index=True), pd.DataFrame(experiment_meta)


def metric_columns(frame_type=FRAME_TYPE):
    bpp_col = f"ave_{frame_type}_frame_bpp"
    psnr_col = f"ave_{frame_type}_frame_psnr"
    return bpp_col, psnr_col


def rd_points(df, experiment, dataset=None, sequence=None, frame_type=FRAME_TYPE):
    bpp_col, psnr_col = metric_columns(frame_type)
    cond = df["experiment"].eq(experiment)
    if dataset is not None:
        cond &= df["dataset"].eq(dataset)
    if sequence is not None:
        cond &= df["sequence"].eq(sequence)

    filtered = df.loc[cond].copy()
    if filtered.empty:
        raise ValueError(f"Sem dados para experiment={experiment}, dataset={dataset}, sequence={sequence}")
    for col in [bpp_col, psnr_col]:
        if col not in filtered.columns:
            raise KeyError(f"Coluna ausente: {col}")
        filtered[col] = pd.to_numeric(filtered[col], errors="coerce")

    grouped = (
        filtered.dropna(subset=["rate_idx", bpp_col, psnr_col])
        .groupby("rate_idx", as_index=False)
        .agg(bpp=(bpp_col, "mean"), psnr=(psnr_col, "mean"))
        .sort_values("rate_idx")
    )
    if len(grouped) < 4:
        raise ValueError(f"Pontos RD insuficientes para {experiment}: {len(grouped)}")
    return grouped["bpp"].to_numpy(), grouped["psnr"].to_numpy()


def safe_bd_rate(anchor_bpp, anchor_psnr, test_bpp, test_psnr, method=BD_RATE_METHOD):
    try:
        return float(bd_rate(anchor_bpp, anchor_psnr, test_bpp, test_psnr, method))
    except Exception as exc:
        print(f"[WARN] BD-rate falhou: {exc}")
        return np.nan


In [ ]:
result_files = discover_result_files(RESULTS_DIR)
df_results, df_experiments = load_all_results(result_files)

print(f"Arquivos carregados: {len(result_files)}")
print(f"Linhas carregadas: {len(df_results):,}")
print(f"Experimentos carregados: {df_results['experiment'].nunique()}")

display(df_experiments.sort_values(["module", "variant", "experiment"], na_position="first").reset_index(drop=True))


In [ ]:
# Garante ordem intra primeiro, depois inter, mas mantém módulos extras ao final se existirem.
found_modules = [m for m in df_experiments["module"].dropna().unique().tolist()]
ordered_modules = [m for m in MODULE_ORDER if m in found_modules]
extra_modules = sorted([m for m in found_modules if m not in ordered_modules], key=lambda x: (not x.startswith("i."), x))
ordered_modules += extra_modules

found_variants = df_experiments["variant"].dropna().unique().tolist()
ordered_variants = [v for v in VARIANT_ORDER if v in found_variants] + [v for v in sorted(found_variants) if v not in VARIANT_ORDER]

print("Módulos ordenados:")
for module in ordered_modules:
    print(f"  - {module}")

print("\nVariantes ordenadas:")
for variant in ordered_variants:
    print(f"  - {variant}")


In [ ]:
def bd_rate_for_experiment(df, experiment, dataset=None, sequence=None, frame_type=FRAME_TYPE):
    anchor_bpp, anchor_psnr = rd_points(df, ANCHOR_NAME, dataset=dataset, sequence=sequence, frame_type=frame_type)
    test_bpp, test_psnr = rd_points(df, experiment, dataset=dataset, sequence=sequence, frame_type=frame_type)
    return safe_bd_rate(anchor_bpp, anchor_psnr, test_bpp, test_psnr)


def build_bd_rate_table(df, modules, variants, dataset=None, sequence=None, frame_type=FRAME_TYPE):
    rows = []
    experiments_available = set(df["experiment"].unique())
    for module in modules:
        row = {"module": module, "type": "intra" if module.startswith("i.") else "inter"}
        for variant in variants:
            experiment = f"{module}__{variant}"
            if experiment not in experiments_available:
                row[variant] = np.nan
                continue
            try:
                row[variant] = bd_rate_for_experiment(
                    df, experiment, dataset=dataset, sequence=sequence, frame_type=frame_type
                )
            except Exception as exc:
                print(f"[WARN] BD-rate indisponível para {experiment}, dataset={dataset}, sequence={sequence}: {exc}")
                row[variant] = np.nan
        rows.append(row)
    table = pd.DataFrame(rows).set_index(["type", "module"])
    table = table.rename(columns={v: VARIANT_LABELS.get(v, v) for v in variants})
    return table

bd_table = build_bd_rate_table(df_results, ordered_modules, ordered_variants, frame_type=FRAME_TYPE)
bd_table


In [ ]:
# Tabela principal: aumento de BD-rate (%) por módulo e variante.
# Positivo = pior que baseline; negativo = melhor que baseline.

def style_bd_table(table):
    return (
        table.style
        .format("{:+.2f}%", na_rep="—")
        .background_gradient(cmap="RdYlGn_r", axis=None)
        .set_caption(f"Aumento de BD-rate vs. {ANCHOR_NAME} — frame_type={FRAME_TYPE}")
    )

style_bd_table(bd_table)


In [ ]:
# Salva a tabela para consulta fora do notebook.
TABLE_OUT = RESULTS_DIR / f"module_sweep_bd_rate_table_{FRAME_TYPE}.csv"
bd_table.to_csv(TABLE_OUT)
print(f"Tabela salva em: {TABLE_OUT}")


In [ ]:
# Tabelas auxiliares por dataset. Útil para identificar se o aumento médio vem de um conjunto específico.
dataset_tables = {}
for dataset in sorted(df_results["dataset"].dropna().unique()):
    dataset_tables[dataset] = build_bd_rate_table(df_results, ordered_modules, ordered_variants, dataset=dataset, frame_type=FRAME_TYPE)
    print(f"\n### Dataset: {dataset}")
    display(style_bd_table(dataset_tables[dataset]))


In [ ]:
def plot_rd_curves_for_module(df, module, variants, dataset=None, frame_type=FRAME_TYPE, ax=None, title_suffix=""):
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 5))

    anchor_bpp, anchor_psnr = rd_points(df, ANCHOR_NAME, dataset=dataset, frame_type=frame_type)
    anchor_style = PLOT_STYLE[ANCHOR_NAME]
    ax.plot(anchor_bpp, anchor_psnr, label="baseline wsilu4", **anchor_style)

    for variant in variants:
        experiment = f"{module}__{variant}"
        if experiment not in set(df["experiment"].unique()):
            continue
        bpp, psnr = rd_points(df, experiment, dataset=dataset, frame_type=frame_type)
        bdr = safe_bd_rate(anchor_bpp, anchor_psnr, bpp, psnr)
        style = PLOT_STYLE.get(variant, {})
        label = f"{VARIANT_LABELS.get(variant, variant)} ({bdr:+.2f}%)"
        ax.plot(bpp, psnr, label=label, **style)

    dataset_label = "todos datasets" if dataset is None else dataset
    ax.set_title(f"{module} — {dataset_label}{title_suffix}")
    ax.set_xlabel("BPP")
    ax.set_ylabel("PSNR (dB)")
    ax.legend(fontsize=9)
    return ax


def plot_module_overview(df, module, variants, frame_type=FRAME_TYPE):
    datasets = [None] + sorted(df["dataset"].dropna().unique().tolist())
    ncols = min(3, len(datasets))
    nrows = math.ceil(len(datasets) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.2 * ncols, 4.6 * nrows), squeeze=False)
    axes_flat = axes.ravel()

    for ax, dataset in zip(axes_flat, datasets):
        plot_rd_curves_for_module(df, module, variants, dataset=dataset, frame_type=frame_type, ax=ax)

    for ax in axes_flat[len(datasets):]:
        ax.axis("off")

    fig.suptitle(f"Curvas RD — módulo {module}", fontsize=16, y=1.01)
    fig.tight_layout()
    return fig


## Curvas RD por módulo

A célula abaixo gera uma figura por módulo. Cada figura mostra:

- painel agregado com todos os datasets;
- painéis separados por dataset;
- baseline `wsilu4`;
- curvas das variantes LUT para o módulo selecionado;
- BD-rate de cada curva na própria legenda.


In [ ]:
for module in ordered_modules:
    fig = plot_module_overview(df_results, module, ordered_variants, frame_type=FRAME_TYPE)
    plt.show()


## Curvas RD individuais por sequência (opcional/extenso)

A célula abaixo é propositalmente mais extensa: ela cria gráficos por sequência para cada módulo. Execute quando quiser investigar outliers vistos nas tabelas por dataset ou na tabela global.


In [ ]:
def plot_sequence_rd_curves_for_module(df, module, variants, dataset, sequence, frame_type=FRAME_TYPE, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4.5))

    anchor_bpp, anchor_psnr = rd_points(df, ANCHOR_NAME, dataset=dataset, sequence=sequence, frame_type=frame_type)
    ax.plot(anchor_bpp, anchor_psnr, label="baseline wsilu4", **PLOT_STYLE[ANCHOR_NAME])

    for variant in variants:
        experiment = f"{module}__{variant}"
        if experiment not in set(df["experiment"].unique()):
            continue
        bpp, psnr = rd_points(df, experiment, dataset=dataset, sequence=sequence, frame_type=frame_type)
        bdr = safe_bd_rate(anchor_bpp, anchor_psnr, bpp, psnr)
        ax.plot(bpp, psnr, label=f"{VARIANT_LABELS.get(variant, variant)} ({bdr:+.2f}%)", **PLOT_STYLE.get(variant, {}))

    ax.set_title(f"{module} — {dataset}/{sequence}")
    ax.set_xlabel("BPP")
    ax.set_ylabel("PSNR (dB)")
    ax.legend(fontsize=8)
    return ax


def plot_all_sequences_for_module(df, module, variants, frame_type=FRAME_TYPE):
    pairs = (
        df[["dataset", "sequence"]]
        .dropna()
        .drop_duplicates()
        .sort_values(["dataset", "sequence"])
        .itertuples(index=False, name=None)
    )
    pairs = list(pairs)
    ncols = 3
    nrows = math.ceil(len(pairs) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.2 * ncols, 4.5 * nrows), squeeze=False)
    axes_flat = axes.ravel()

    for ax, (dataset, sequence) in zip(axes_flat, pairs):
        plot_sequence_rd_curves_for_module(df, module, variants, dataset, sequence, frame_type=frame_type, ax=ax)

    for ax in axes_flat[len(pairs):]:
        ax.axis("off")

    fig.suptitle(f"Curvas RD por sequência — módulo {module}", fontsize=16, y=1.005)
    fig.tight_layout()
    return fig

# Descomente para gerar os gráficos por sequência para todos os módulos.
# for module in ordered_modules:
#     fig = plot_all_sequences_for_module(df_results, module, ordered_variants, frame_type=FRAME_TYPE)
#     plt.show()


## Ranking dos maiores aumentos de BD-rate

A célula abaixo transforma a tabela principal para formato longo e lista os maiores aumentos.


In [ ]:
bd_long = (
    bd_table
    .reset_index()
    .melt(id_vars=["type", "module"], var_name="variant", value_name="bd_rate_increase_pct")
    .sort_values("bd_rate_increase_pct", ascending=False)
)

display(bd_long.head(30).style.format({"bd_rate_increase_pct": "{:+.2f}%"}))
